In [1]:
import pandas as pd
import os
import numpy as np
from gseapy import Biomart

In [2]:
df_mapping = pd.read_csv('../related_files/mouse_gene_name_old_new_mapping.csv')
bm = Biomart()
# note the dataset and attribute names are different
m2h = bm.query(dataset='mmusculus_gene_ensembl',
               attributes=['ensembl_gene_id','external_gene_name',
                           'hsapiens_homolog_ensembl_gene',
                           'hsapiens_homolog_associated_gene_name'])

In [3]:
for sample in os.listdir('../new_mageck/'):
    if not sample.startswith('.'):
        gene_summary_path = f'../new_mageck/{sample}/{sample}.gene_summary.txt' 
        df = pd.read_csv(gene_summary_path, sep='\t', index_col=0) 
        mapping = dict(zip(df_mapping['query'], df_mapping['symbol']))
        df['gene'] = [mapping[i.split('_')[0]] if i in mapping.keys() else i.split('_')[0] for i in df.index]

        mapping = dict(zip(m2h['external_gene_name'], m2h['hsapiens_homolog_associated_gene_name']))
        df['human homolog'] = [mapping[i] if i in mapping.keys() else np.nan for i in df['gene'].tolist()]

        df['lfc'] = df['neg|lfc']
        df['fdr'] = df.apply(lambda x:x['neg|fdr'] if x['lfc']<0 else x['pos|fdr'], axis=1)
        df['pvalue'] = df.apply(lambda x:x['neg|p-value'] if x['lfc']<0 else x['pos|p-value'], axis=1)

        df['hits'] = df.apply(lambda x:-1 if ((x['neg|fdr']<0.1) & (x['neg|lfc']<0)) else 1 if ((x['pos|fdr']<0.1) & (x['pos|lfc']>0)) else 0, axis=1)
        df.to_csv(gene_summary_path, sep='\t')

In [9]:
file_path = '../roc_auc/mageck_output/'
for sample in os.listdir(file_path):
    if not sample.startswith('.'):
        gene_summary_path = f'{file_path}{sample}/{sample}.gene_summary.txt' 
        df = pd.read_csv(gene_summary_path, sep='\t', index_col=0) 
        mapping = dict(zip(df_mapping['query'], df_mapping['symbol']))
        df['gene'] = [mapping[i.split('_')[0]] if i in mapping.keys() else i.split('_')[0] for i in df.index]
        
        mapping = dict(zip(m2h['external_gene_name'], m2h['hsapiens_homolog_associated_gene_name']))
        df['human homolog'] = [mapping[i] if i in mapping.keys() else np.nan for i in df['gene'].tolist()]
       
        df['lfc'] = df['neg|lfc']
        df['fdr'] = df.apply(lambda x:x['neg|fdr'] if x['lfc']<0 else x['pos|fdr'], axis=1)
        df['pvalue'] = df.apply(lambda x:x['neg|p-value'] if x['lfc']<0 else x['pos|p-value'], axis=1)

        df['hits'] = df.apply(lambda x:-1 if ((x['neg|fdr']<0.1) & (x['neg|lfc']<0)) else 1 if ((x['pos|fdr']<0.1) & (x['pos|lfc']>0)) else 0, axis=1)
        df.to_csv(gene_summary_path, sep='\t')